# FinSight: Financial Exploratory Data Analysis (EDA)
### Data Analytics & Cash Flow Intelligence
This notebook provides in-depth exploratory analysis of 70,000+ financial transaction records,
examining revenue trends, expenditure patterns, channel reliability, and statistical outliers.

In [1]:
import duckdb
con = duckdb.connect()
con.execute("CREATE TABLE transactions AS SELECT * FROM read_csv_auto('../backend/data/processed/transactions_cleaned.csv')")
print(f"Loaded {con.execute('SELECT COUNT(*) FROM transactions').fetchone()[0]:,} cleaned transactions.")

## 1. High-Level Dataset Dimensions and Descriptive Statistics

In [2]:
con.execute('''
    SELECT 
        COUNT(*) AS total_records,
        COUNT(DISTINCT customer_id) AS unique_customers,
        ROUND(AVG(amount), 2) AS mean_amount,
        ROUND(MEDIAN(amount), 2) AS median_amount,
        ROUND(STDDEV(amount), 2) AS stddev_amount,
        ROUND(MIN(amount), 2) AS min_amount,
        ROUND(MAX(amount), 2) AS max_amount
    FROM transactions
''').fetchall()

## 2. Cash Flow Engine: Realized Income vs Expenses

In [3]:
con.execute('''
    SELECT 
        transaction_type,
        COUNT(*) AS settled_orders,
        ROUND(SUM(amount), 2) AS total_settled_amount,
        ROUND(AVG(amount), 2) AS avg_ticket_size
    FROM transactions
    WHERE status = 'Completed'
    GROUP BY transaction_type
''').fetchall()

## 3. Month-over-Month Cash Flow & Revenue Acceleration

In [4]:
con.execute('''
    SELECT 
        strftime(transaction_date::DATE, '%Y-%m') AS month,
        ROUND(SUM(CASE WHEN transaction_type = 'Income' THEN amount ELSE 0 END), 2) AS gross_revenue,
        ROUND(SUM(CASE WHEN transaction_type = 'Expense' THEN amount ELSE 0 END), 2) AS gross_expense,
        ROUND(SUM(CASE WHEN transaction_type = 'Income' THEN amount ELSE -amount END), 2) AS net_cash_flow
    FROM transactions
    WHERE status = 'Completed'
    GROUP BY strftime(transaction_date::DATE, '%Y-%m')
    ORDER BY month ASC
''').fetchall()

## 4. Payment Gateway Reliability & Success Rates

In [5]:
con.execute('''
    SELECT 
        payment_method,
        COUNT(*) AS total_attempts,
        ROUND(100.0 * SUM(CASE WHEN status = 'Completed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS success_rate_pct,
        ROUND(100.0 * SUM(CASE WHEN status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS failure_rate_pct,
        ROUND(SUM(CASE WHEN status = 'Completed' THEN amount ELSE 0 END), 2) AS total_settled_value
    FROM transactions
    GROUP BY payment_method
    ORDER BY total_attempts DESC
''').fetchall()

## 5. Statistical Anomaly & Outlier Identification (Tukey's IQR)

In [6]:
con.execute('''
    WITH cat_stats AS (
        SELECT 
            category,
            QUANTILE_CONT(amount, 0.25) AS q1,
            QUANTILE_CONT(amount, 0.75) AS q3
        FROM transactions
        WHERE status = 'Completed'
        GROUP BY category
    )
    SELECT 
        t.transaction_id,
        t.transaction_date,
        t.category,
        t.amount,
        ROUND((c.q3 + 1.5 * (c.q3 - c.q1)), 2) AS upper_fence,
        ROUND(t.amount - (c.q3 + 1.5 * (c.q3 - c.q1)), 2) AS excess_amount
    FROM transactions t
    JOIN cat_stats c ON t.category = c.category
    WHERE t.status = 'Completed' AND t.amount > (c.q3 + 3.0 * (c.q3 - c.q1))
    ORDER BY excess_amount DESC
    LIMIT 15
''').fetchall()